# Análisis de Datos Exploratorios (EDA)

## Importar Datos
**Import Data**

In [1]:
import pandas as pd
import json
import numpy as np

df = pd.read_csv("../../data/transactions_sample.csv", low_memory=False)

print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nFirst 3 rows:\n", df.head(3).to_string())

Shape: (204000, 13)

Dtypes:
 transaction_id          int64
merchant_id             int64
transaction_date       object
amount                 object
status                 object
channel                object
cancellation_reason    object
reference_date         object
fla_churn90             int64
last_complaint_date    object
segment                object
mcc                     int64
dat_process            object
dtype: object

First 3 rows:
    transaction_id  merchant_id transaction_date amount    status channel cancellation_reason reference_date  fla_churn90 last_complaint_date segment   mcc dat_process
0               1     10007653       2025-09-11  18,50  approved     pos                 NaN     2025-09-30            0                 NaN     SMB  4789  2025-09-11
1               2     10009541       2024-03-16  16,81  approved    ecom                 NaN     2025-09-30            0          2025-05-14     SMB  7995  2024-03-16
2               3     10006986       30/04/2024  

## Valores Nulos y Cardinalidades
**Nulls and Cardinalities**

In [ ]:
print("=== NULLS ===")
print(df.isnull().sum().sort_values(ascending=False))

print("\n=== UNIQUE COUNTS ===")
print(df.nunique())

# Low-cardinality cols, show all values
for col in ["status", "channel", "segment", "fla_churn90"]:
    print(f"\n{col}:\n", df[col].value_counts(dropna=False))

## Explorar *amount* & *transaction_date* 
**Explore amount & transaction_date**

In [ ]:
# amount
print("=== AMOUNT ===")
print("Sample values:", df["amount"].sample(10).tolist())
df["amount_numeric"] = pd.to_numeric(df["amount"], errors="coerce")
print("Parse failures (NaN):", df["amount_numeric"].isna().sum())
print("Negatives:", (df["amount_numeric"] < 0).sum())
print("Zeros:", (df["amount_numeric"] == 0).sum())
print("Stats:\n", df["amount_numeric"].describe())

# transaction_date
print("\n=== TRANSACTION_DATE ===")
print("Sample values:", df["transaction_date"].sample(10).tolist())
df["tx_date_parsed"] = pd.to_datetime(df["transaction_date"], errors="coerce")
print("Parse failures:", df["tx_date_parsed"].isna().sum())
print("Date range:", df["tx_date_parsed"].min(), "to", df["tx_date_parsed"].max())

## Verificación de Fugas 
**Leakage Check**

In [ ]:
# Are there transactions dated AFTER the reference_date snapshot
df["ref_date"] = pd.to_datetime(df["reference_date"], errors="coerce")
future_leak = df["tx_date_parsed"] > df["ref_date"]
print("Transactions after reference_date:", future_leak.sum())

# Does cancellation_reason only appear for right statuses
print("\n=== cancellation_reason by status ===")
print(df.groupby("status")["cancellation_reason"].apply(
    lambda x: x.notna().sum()
))

# Does last_complaint_date leak target
print("\n=== last_complaint_date fill rate by churn label ===")
print(df.groupby("fla_churn90")["last_complaint_date"].apply(
    lambda x: x.notna().mean()
))

Transactions after reference_date: 23014

=== cancellation_reason by status ===
status
approved    11609
denied       1336
reversed      274
Name: cancellation_reason, dtype: int64

=== last_complaint_date fill rate by churn label ===
fla_churn90
0    0.282577
1    0.338718
Name: last_complaint_date, dtype: float64


## Duplicados 
**Duplicates**

In [5]:
print("Duplicate transaction_ids:", df["transaction_id"].duplicated().sum())
print("Duplicate full rows:", df.duplicated().sum())

# Are merchant_ids consistent with the merchants_context.json?
with open("../../data/merchants_context.json") as f:
    merchants = json.load(f)

merchant_ids_json = {m["merchant_id"] for m in merchants}
merchant_ids_csv  = set(df["merchant_id"].unique())

print("\nMerchants in CSV:", len(merchant_ids_csv))
print("Merchants in JSON:", len(merchant_ids_json))
print("In CSV but not JSON:", len(merchant_ids_csv - merchant_ids_json))
print("In JSON but not CSV:", len(merchant_ids_json - merchant_ids_csv))

Duplicate transaction_ids: 0
Duplicate full rows: 0

Merchants in CSV: 9982
Merchants in JSON: 500
In CSV but not JSON: 9483
In JSON but not CSV: 1


## Distribución del Target: Desbalance de Clases
**Target Distribution: Class Imbalance**

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.parte1_pandas import load_clean

df_clean = load_clean("../../data/transactions_sample.csv")

# One row per merchant for target analysis
merchant_level = df_clean.drop_duplicates("merchant_id")[["merchant_id","fla_churn90","segment","mcc"]].copy()

churn_counts = merchant_level["fla_churn90"].value_counts()
print("=== TARGET DISTRIBUTION (merchant level) ===")
print(churn_counts)
print(f"\nChurn rate: {churn_counts[1]/len(merchant_level):.2%}")
print(f"Class ratio 0:1 = {churn_counts[0]/churn_counts[1]:.1f}:1, imbalanced dataset")

print("\n=== CHURN RATE BY SEGMENT ===")
seg_churn = merchant_level.groupby("segment", observed=True)["fla_churn90"].agg(["mean","count"])
seg_churn.columns = ["churn_rate","n_merchants"]
print(seg_churn.round(4))
print("\n- SMB churns ~3x more than Enterprise. Segment is a strong prior.")

## Leakage Temporal: Transacciones Después de reference_date
**Temporal Leakage: Transactions After reference_date**

El dataset abarca 2024-01-01 a 2025-12-31 pero `reference_date = 2025-09-30`. Las transacciones fechadas DESPUÉS del snapshot son **datos futuros** relativos al punto de predicción: usarlas causaría leakage (un merchant activo claramente no está churning).

In [ ]:
ref_date = df_clean["reference_date"].max()
print(f"Reference date: {ref_date.date()}")

post_ref = df_clean["transaction_date"] > ref_date
print(f"\nTransactions AFTER reference_date: {post_ref.sum():,} ({post_ref.mean():.1%})")
print(f"Transactions UP TO reference_date:  {(~post_ref).sum():,} ({(~post_ref).mean():.1%})")

# These post-ref tx belong to which merchant types?
print("\nChurn rate for merchants with post-ref activity:")
merchants_with_future = df_clean.loc[post_ref, "merchant_id"].unique()
future_churn = merchant_level.set_index("merchant_id")["fla_churn90"].reindex(merchants_with_future)
print(f"  {future_churn.mean():.2%} churn, much lower (active merchants rarely churn)")
print("\nConclusion: ALL features must be built from tx where transaction_date <= reference_date")

## Volumen y TPV por Canal y Segmento
**Volume & TPV by Channel and Segment**

In [ ]:
df_pre = df_clean[df_clean["transaction_date"] <= ref_date].copy()

print("=== CHANNEL BREAKDOWN (pre-reference transactions) ===")
ch = df_pre.groupby("channel", observed=True).agg(
    n_tx=("transaction_id","count"),
    tpv=("amount","sum"),
    approval_rate=("status", lambda x: (x == "approved").mean())
).assign(pct_volume=lambda x: x["n_tx"]/x["n_tx"].sum())
print(ch.round(4))
print("\n- POS dominates volume (55%); ecom has slightly lower approval rate.")

print("\n=== AMOUNT DISTRIBUTION BY SEGMENT ===")
seg_amount = df_pre.groupby("segment", observed=True)["amount"].describe()[["mean","50%","std","max"]]
print(seg_amount.round(2))
print("\n- Enterprise amounts are ~33x larger than SMB. Log-transform needed for modeling.")

## Vista Previa de Features por Merchant
**Merchant-Level Feature Preview**

Construcción de features a nivel merchant (una fila por merchant) usando solo transacciones previas a reference_date. Esta es la granularidad que necesita el modelo de churn.

In [9]:
window_3m = ref_date - pd.Timedelta(days=90)

def build_merchant_features(df, ref, window):
    """Build one row per merchant with aggregate features."""
    pre = df[df["transaction_date"] <= ref]
    recent = pre[pre["transaction_date"] >= window]

    # All-time features (up to reference)
    all_agg = pre.groupby("merchant_id", observed=True).agg(
        tpv_total=("amount","sum"),
        n_tx_total=("transaction_id","count"),
        approval_rate_total=("status", lambda x: (x=="approved").mean()),
    )
    # Recent 3m features
    recent_agg = recent.groupby("merchant_id", observed=True).agg(
        tpv_3m=("amount","sum"),
        n_tx_3m=("transaction_id","count"),
        approval_rate_3m=("status", lambda x: (x=="approved").mean()),
        pct_ecom_3m=("channel", lambda x: (x=="ecom").mean()),
    )
    # Complaint feature (only valid pre-reference dates)
    complaint_raw = df.groupby("merchant_id", observed=True)["last_complaint_date"].first().reset_index()
    complaint_raw["lcd_valid"] = complaint_raw["last_complaint_date"].where(
        complaint_raw["last_complaint_date"] <= ref
    )
    complaint_raw["days_since_complaint"] = (ref - complaint_raw["lcd_valid"]).dt.days
    complaint_raw = complaint_raw.set_index("merchant_id")[["days_since_complaint"]]

    # Metadata
    meta = df.drop_duplicates("merchant_id").set_index("merchant_id")[["segment","mcc","fla_churn90"]]

    feats = all_agg.join(recent_agg, how="left").join(complaint_raw, how="left").join(meta, how="left")
    feats["tpv_ratio_3m"] = feats["tpv_3m"] / feats["tpv_total"].replace(0, float("nan"))
    feats["log_tpv_total"] = feats["tpv_total"].clip(lower=0.01).apply(lambda x: __import__("math").log(x))
    return feats.reset_index()

merchant_feats = build_merchant_features(df_clean, ref_date, window_3m)
print(f"Merchant feature matrix: {merchant_feats.shape}")
print(f"Churners: {merchant_feats['fla_churn90'].sum()} / {len(merchant_feats)}")
print("\nFeature preview:")
print(merchant_feats[["merchant_id","tpv_3m","approval_rate_3m","days_since_complaint","segment","fla_churn90"]].head(5).to_string())

# Correlation of numeric features with target
print("\n=== PEARSON CORRELATION WITH TARGET ===")
num_feats = ["tpv_total","tpv_3m","n_tx_3m","approval_rate_3m","pct_ecom_3m","tpv_ratio_3m","days_since_complaint","log_tpv_total"]
corr = merchant_feats[num_feats + ["fla_churn90"]].corr()["fla_churn90"].drop("fla_churn90")
print(corr.sort_values().round(4))

Merchant feature matrix: (9967, 14)
Churners: 871 / 9967

Feature preview:
   merchant_id  tpv_3m  approval_rate_3m  days_since_complaint segment  fla_churn90
0     10000000   53.40          0.000000                  69.0     SMB            0
1     10000001  275.72          0.666667                   NaN     SMB            0
2     10000002   15.54          1.000000                   NaN     SMB            0
3     10000003   23.59          1.000000                   NaN     SMB            0
4     10000004     NaN               NaN                  76.0     SMB            0

=== PEARSON CORRELATION WITH TARGET ===
log_tpv_total          -0.0446
tpv_3m                 -0.0177
tpv_total              -0.0165
n_tx_3m                -0.0043
days_since_complaint   -0.0016
approval_rate_3m       -0.0007
pct_ecom_3m             0.0011
tpv_ratio_3m            0.0086
Name: fla_churn90, dtype: float64


## Tendencias Mensuales de Transacciones
**Monthly Transaction Trends**

In [ ]:
df_clean["month"] = df_clean["transaction_date"].dt.to_period("M")
monthly = df_clean.groupby("month").agg(
    n_tx=("transaction_id","count"),
    tpv=("amount","sum"),
    approval_rate=("status", lambda x: (x=="approved").mean())
).reset_index()

print("=== MONTHLY VOLUME (all merchants, all periods) ===")
print(monthly.tail(12).to_string(index=False))
print("\n- ~8,000-8,500 tx/month, stable volume. No strong seasonal pattern.")
print("- Post-reference months (Oct-Dec 2025) show normal activity, confirming they are future data.")

## Resumen EDA: Hallazgos Clave para el Modelado
**EDA Summary: Key Findings for Modeling**

| Hallazgo / Finding | Impacto en el Modelo / Impact on Model |
|---|---|
| **Tasa de churn 8.75%** (imbalance 1:10.4) | Usar PR-AUC, no accuracy; considerar `scale_pos_weight` en LightGBM |
| **Existen transacciones después de reference_date** | Filtrar `transaction_date <= reference_date` antes de construir features |
| **`cancellation_reason`** rellena 92% para churners, 0% para no-churners | Leakage directo, excluir completamente |
| **`last_complaint_date`** > reference_date en 3,454 filas | Leakage temporal, capar en reference_date |
| **SMB churnea al 9.5%**, Enterprise al 3.2% | `segment` es un feature categórico fuerte |
| **Amount de R$0.5 a R$67k** con skew hacia la derecha | Log-transform en agregaciones numéricas |
| **Problemas de calidad T3-T5** | Manejados en `load_clean`, documentar en DECISIONS |